# LeetCode 3238 — Find the Number of Winning Players bằng Python

Link đề: https://leetcode.com/problems/find-the-number-of-winning-players/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Counting

Skeleton LeetCode:

```python
class Solution:
    def winningPlayerCount(self, n: int, pick: List[List[int]]) -> int:
        ...
```

Ghi chú: mọi đáp án trong notebook đã được kiểm chứng bằng code (kể cả đối chiếu với bản vét cạn). Ba ví dụ chính lấy từ đề; những ví dụ tự dựng thêm có ghi rõ "tự dựng" — nếu muốn khớp từng chữ thì nên đối chiếu lại đề gốc. Trong notebook ta viết `list[list[int]]` của Python mới thay cho `List[List[int]]` (LeetCode đã `import` sẵn `List`).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: có `n` người chơi đánh số `0..n-1`; mỗi phần tử `pick[i] = [x, y]` nghĩa là người `x` nhặt một quả bóng màu `y`; người `i` **thắng** nếu nhặt được **nhiều hơn `i` quả cùng một màu** (tức ít nhất `i + 1` quả cùng màu); đếm số người thắng.
2. Nắm các công cụ Python: danh sách hai chiều `cnt[x][y]`, **cách tạo đúng** `[[0] * 11 for _ in range(n)]`, `set`, `collections.Counter`, `defaultdict`, bóc tách `for x, y in pick`, và `sum` trên danh sách cờ.
3. Đi từ cách **vét cạn đếm cho từng (người, màu)** → **bảng đếm 2 chiều + `set` người thắng** → **`Counter` trên cặp `(x, y)`** → **số lượng lớn nhất theo từng người** → **mảng cờ `won[]`**.
4. Hiểu mấu chốt: đây là bài **đếm theo nhóm** — nhóm là cặp `(người, màu)`; một người thắng ⇔ **có ít nhất một màu** mà số quả của màu đó `> chỉ số người`; và phải **đếm người**, không đếm cặp, nên một người có nhiều màu "đủ điều kiện" chỉ được tính **một lần**.
5. Tránh các lỗi kinh điển: đếm trùng người (cộng `1` mỗi khi một màu đủ điều kiện), dùng `>=` thay vì `>`, đếm **tổng số bóng** thay vì theo **từng màu**, và tạo bảng 2 chiều bằng `[[0] * 11] * n` (các hàng trỏ chung một danh sách).

## 2. Đề bài & ví dụ

Cho số nguyên `n` là số người chơi và mảng 2 chiều `pick`, trong đó `pick[i] = [x_i, y_i]` nghĩa là người chơi `x_i` đã nhặt một quả bóng màu `y_i`.

Người chơi `i` **thắng** nếu nhặt được **nhiều hơn `i` quả bóng cùng màu**. Nói cách khác:

- Người `0` thắng nếu nhặt **bất kỳ** quả bóng nào.
- Người `1` thắng nếu nhặt ít nhất `2` quả **cùng màu**.
- ...
- Người `i` thắng nếu nhặt ít nhất `i + 1` quả **cùng màu**.

Trả về **số người thắng**. Lưu ý nhiều người có thể cùng thắng.

**Ví dụ 1**

- Input: `n = 4, pick = [[0,0],[1,0],[1,0],[2,1],[2,1],[2,0]]`
- Output: `2`
- Giải thích: người `0` và người `1` thắng; người `2` (cần `3` quả cùng màu, chỉ có `2` quả màu `1` và `1` quả màu `0`) và người `3` (không nhặt gì) không thắng.

**Ví dụ 2**

- Input: `n = 5, pick = [[1,1],[1,2],[1,3],[1,4]]`
- Output: `0`
- Giải thích: người `1` nhặt `4` quả nhưng mỗi quả một màu khác nhau nên không đủ `2` quả cùng màu. Không ai thắng.

**Ví dụ 3**

- Input: `n = 5, pick = [[1,1],[2,4],[2,4],[2,4]]`
- Output: `1`
- Giải thích: người `2` thắng vì nhặt `3` quả màu `4`.

**Ràng buộc:** `2 <= n <= 10`; `1 <= pick.length <= 100`; `pick[i].length == 2`; `0 <= x_i <= n - 1`; `0 <= y_i <= 10`.

In [1]:
examples = [
    (4, [[0, 0], [1, 0], [1, 0], [2, 1], [2, 1], [2, 0]], 2),
    (5, [[1, 1], [1, 2], [1, 3], [1, 4]], 0),
    (5, [[1, 1], [2, 4], [2, 4], [2, 4]], 1),
]
for n, pick, expected in examples:
    print(f"n={n}, pick={pick} -> kỳ vọng {expected}")

n=4, pick=[[0, 0], [1, 0], [1, 0], [2, 1], [2, 1], [2, 0]] -> kỳ vọng 2
n=5, pick=[[1, 1], [1, 2], [1, 3], [1, 4]] -> kỳ vọng 0
n=5, pick=[[1, 1], [2, 4], [2, 4], [2, 4]] -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Bóc tách cặp: `for x, y in pick`

Mỗi phần tử của `pick` là một danh sách `[x, y]` gồm hai số. Python cho phép **bóc tách** (unpacking) ngay trong `for`: `for x, y in pick:` gán `x = phần_tử[0]`, `y = phần_tử[1]`. Gọn hơn `p[0]`, `p[1]` và dễ đọc hơn.

In [2]:
pick = [[0, 0], [1, 0], [1, 0]]
for x, y in pick:
    print("người", x, "nhặt bóng màu", y)

người 0 nhặt bóng màu 0
người 1 nhặt bóng màu 0
người 1 nhặt bóng màu 0


### 3.2. Bảng đếm 2 chiều: `cnt[x][y]`

`cnt[x][y]` lưu "người `x` đã nhặt bao nhiêu quả màu `y`". Có `n` người và `11` màu (`0..10`), nên bảng có `n` hàng, `11` cột. **Cách tạo đúng**: `[[0] * 11 for _ in range(n)]` (mỗi hàng là một danh sách **mới**). Mỗi lần nhặt: `cnt[x][y] += 1`.

In [3]:
n = 3
cnt = [[0] * 11 for _ in range(n)]
for x, y in [[0, 0], [1, 0], [1, 0], [2, 4]]:
    cnt[x][y] += 1
print(cnt[0][:5], cnt[1][:5], cnt[2][:5])     # [1, 0, 0, 0, 0] [2, 0, 0, 0, 0] [0, 0, 0, 0, 1]
print(len(cnt), len(cnt[0]))                  # 3 11

[1, 0, 0, 0, 0] [2, 0, 0, 0, 0] [0, 0, 0, 0, 1]
3 11


### 3.3. `set` để đếm "mỗi người một lần"

`set` là tập hợp **không trùng**: thêm cùng một phần tử nhiều lần vẫn chỉ có **một** bản. `s.add(x)` thêm, `len(s)` là số phần tử. Rất hợp cho "danh sách các người thắng" vì một người có thể đạt điều kiện nhiều lần nhưng chỉ nên tính một lần.

In [4]:
winners = set()
for x in [1, 0, 1, 1, 0]:
    winners.add(x)
print(winners, len(winners))      # {0, 1} 2

{0, 1} 2


### 3.4. `Counter` đếm số lần xuất hiện

`from collections import Counter`; `Counter(iterable)` đếm mỗi phần tử xuất hiện bao nhiêu lần, trả một `dict`-gần-như. Khóa phải **băm được**: danh sách `[x, y]` thì **không**, nhưng bộ `(x, y)` (tuple) thì có. Dùng `map(tuple, pick)` để đổi từng `[x, y]` thành `(x, y)`.

In [5]:
from collections import Counter, defaultdict

pick = [[0, 0], [1, 0], [1, 0], [2, 1], [2, 1], [2, 0]]
c = Counter(map(tuple, pick))
print(c)                          # Counter({(1, 0): 2, (2, 1): 2, (0, 0): 1, (2, 0): 1})
print(c[(1, 0)], c[(9, 9)])       # 2 0 (khóa không có thì là 0)

Counter({(1, 0): 2, (2, 1): 2, (0, 0): 1, (2, 0): 1})
2 0


### 3.5. `defaultdict` lồng và `max` theo từng người

`defaultdict(Counter)` tạo ra từ điển mà khóa mới tự có một `Counter` rỗng: `by_player[x][y] += 1` đếm màu `y` của người `x`. Sau đó `max(counter.values(), default=0)` cho số quả cùng màu **nhiều nhất** của người đó (`default=0` khi không nhặt gì).

In [6]:
by_player = defaultdict(Counter)
for x, y in pick:
    by_player[x][y] += 1
print({x: dict(c) for x, c in by_player.items()})            # {0: {0: 1}, 1: {0: 2}, 2: {1: 2, 0: 1}}
print([max(by_player[i].values(), default=0) for i in range(4)])    # [1, 2, 2, 0]

{0: {0: 1}, 1: {0: 2}, 2: {1: 2, 0: 1}}
[1, 2, 2, 0]


### 3.6. Danh sách cờ và `sum` của `True`

`won = [False] * n` rồi `won[x] = True` đánh dấu người thắng. `sum(won)` đếm số `True` (vì `True == 1`, `False == 0`). Cũng có thể `sum(1 for ... if ...)`.

In [7]:
won = [False] * 4
won[0] = True
won[1] = True
print(won, sum(won))              # [True, True, False, False] 2
print(sum(1 for w in won if w))   # 2

[True, True, False, False] 2
2


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ 1: `n = 4`, `pick = [[0,0],[1,0],[1,0],[2,1],[2,1],[2,0]]`. Lập bảng **số quả mỗi màu của từng người**:

| người `i` | màu 0 | màu 1 | cần (`> i`) | thắng? |
|---|---|---|---|---|
| 0 | 1 | 0 | `> 0` (≥ 1) | **có** (1 quả màu 0) |
| 1 | 2 | 0 | `> 1` (≥ 2) | **có** (2 quả màu 0) |
| 2 | 1 | 2 | `> 2` (≥ 3) | không (lớn nhất là 2) |
| 3 | 0 | 0 | `> 3` (≥ 4) | không |

Có `2` người thắng. Quan sát: ta chỉ cần **đếm theo cặp (người, màu)** rồi so sánh với `chỉ số người`.

Ví dụ 2 (tại sao đếm theo **màu**): người `1` nhặt `4` quả màu `1, 2, 3, 4` ⇒ mỗi màu `1` quả, cần `2` ⇒ không thắng; nếu đếm tổng `4 > 1` thì sẽ sai.

In [8]:
n, pick = 4, [[0, 0], [1, 0], [1, 0], [2, 1], [2, 1], [2, 0]]
for i in range(n):
    per_color = {}
    for x, y in pick:
        if x == i:
            per_color[y] = per_color.get(y, 0) + 1
    best = max(per_color.values(), default=0)
    print(f"người {i}: đếm theo màu = {per_color}, lớn nhất = {best}, cần > {i} -> {'THẮNG' if best > i else 'không'}")

người 0: đếm theo màu = {0: 1}, lớn nhất = 1, cần > 0 -> THẮNG
người 1: đếm theo màu = {0: 2}, lớn nhất = 2, cần > 1 -> THẮNG
người 2: đếm theo màu = {1: 2, 0: 1}, lớn nhất = 2, cần > 2 -> không
người 3: đếm theo màu = {}, lớn nhất = 0, cần > 3 -> không


## 5. Cách 1 — Brute Force / baseline: đếm trực tiếp cho từng (người, màu)

**Ý tưởng.** Với mỗi người `i` trong `0..n-1` và mỗi màu `c` trong `0..10`: đếm xem cặp `[i, c]` xuất hiện bao nhiêu lần trong `pick` (bằng `pick.count([i, c])`). Nếu **có màu nào** đếm được `> i` thì người `i` thắng — cộng `1` và **thoát** vòng lặp màu (`break`) để không đếm trùng.

In [9]:
def winning_player_count_brute(n: int, pick: list[list[int]]) -> int:
    ans = 0
    for i in range(n):                      # từng người chơi
        for c in range(11):                 # từng màu 0..10
            if pick.count([i, c]) > i:      # số lần người i nhặt màu c
                ans += 1
                break                       # thắng rồi, không xét màu khác (tránh đếm trùng)
    return ans


for n, pick, expected in examples:
    print(winning_player_count_brute(n, pick), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
0 (kỳ vọng 0 )
1 (kỳ vọng 1 )


**Độ phức tạp (Cách 1).** `n` người × `11` màu × `pick.count` mỗi lần quét `m = len(pick)` phần tử ⇒ **Time `O(n · 11 · m)`**, với `n <= 10`, `m <= 100` khoảng `11 000` bước. **Space `O(1)`**.

**Nút thắt:** mỗi lần `pick.count` quét lại toàn bộ `pick`. Nên **quét `pick` một lần** và ghi dồn kết quả vào bảng đếm.

## 6. Ý tưởng thuật toán / bước đột phá

**Quét một lượt + bảng đếm.** Với mỗi `[x, y]` ta tăng `cnt[x][y]`. Ngay khi `cnt[x][y] > x` ta biết người `x` đã thắng, ghi `x` vào một **`set`** (để dù thắng ở nhiều màu hay vượt ngưỡng nhiều lần vẫn chỉ tính **một** người). Đáp án là `len(set)`.

Mỗi phần tử `pick` được xử lý **một lần**, `O(1)` ⇒ tổng `O(m)`.

In [10]:
n, pick = 4, [[0, 0], [1, 0], [1, 0], [2, 1], [2, 1], [2, 0]]
cnt = [[0] * 11 for _ in range(n)]
winners = set()
for x, y in pick:
    cnt[x][y] += 1
    flag = cnt[x][y] > x
    if flag:
        winners.add(x)
    print(f"nhặt [{x},{y}]: cnt[{x}][{y}] = {cnt[x][y]}, > {x}? {flag}, winners = {sorted(winners)}")
print("đáp án:", len(winners))

nhặt [0,0]: cnt[0][0] = 1, > 0? True, winners = [0]
nhặt [1,0]: cnt[1][0] = 1, > 1? False, winners = [0]
nhặt [1,0]: cnt[1][0] = 2, > 1? True, winners = [0, 1]
nhặt [2,1]: cnt[2][1] = 1, > 2? False, winners = [0, 1]
nhặt [2,1]: cnt[2][1] = 2, > 2? False, winners = [0, 1]
nhặt [2,0]: cnt[2][0] = 1, > 2? False, winners = [0, 1]
đáp án: 2


## 7. Các cách tiếp theo

### Cách 2 — Bảng đếm 2 chiều + `set` người thắng

**Giải thích từng đoạn.**

1. `cnt = [[0] * 11 for _ in range(n)]`: `n` hàng, mỗi hàng `11` ô, **mỗi hàng là một danh sách riêng**.
2. `for x, y in pick:` — duyệt từng lần nhặt.
3. `cnt[x][y] += 1`, rồi `if cnt[x][y] > x: s.add(x)`.
4. `return len(s)`.

In [11]:
def winning_player_count_table(n: int, pick: list[list[int]]) -> int:
    cnt = [[0] * 11 for _ in range(n)]       # cnt[x][y]: số quả màu y của người x
    s = set()                                # các người đã thắng
    for x, y in pick:
        cnt[x][y] += 1
        if cnt[x][y] > x:
            s.add(x)
    return len(s)


for n, pick, expected in examples:
    print(winning_player_count_table(n, pick), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
0 (kỳ vọng 0 )
1 (kỳ vọng 1 )


In [12]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionTable:
    def winningPlayerCount(self, n: int, pick: list[list[int]]) -> int:
        cnt = [[0] * 11 for _ in range(n)]
        s = set()
        for x, y in pick:
            cnt[x][y] += 1
            if cnt[x][y] > x:
                s.add(x)
        return len(s)


print(SolutionTable().winningPlayerCount(4, [[0, 0], [1, 0], [1, 0], [2, 1], [2, 1], [2, 0]]))    # 2

2


**Độ phức tạp Cách 2:** **Time `O(m + n · M)`** với `M = 11` màu (tạo bảng `n × 11`) và `m = len(pick)`; **Space `O(n · M)`**.

### Cách 3 — `Counter` trên các cặp `(x, y)`

**Giải thích.** Thay bảng bằng `Counter(map(tuple, pick))`: khóa là cặp `(x, y)`, giá trị là số lần. Rồi lọc những cặp có `số_lần > x`, lấy **tập các `x`** bằng set comprehension `{x for (x, y), c in counter.items() if c > x}`. Không cần biết trước số màu.

In [13]:
def winning_player_count_counter(n: int, pick: list[list[int]]) -> int:
    counter = Counter(map(tuple, pick))                          # (người, màu) -> số lần
    winners = {x for (x, y), c in counter.items() if c > x}      # tập các người có ít nhất một màu đủ điều kiện
    return len(winners)


for n, pick, expected in examples:
    print(winning_player_count_counter(n, pick), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
0 (kỳ vọng 0 )
1 (kỳ vọng 1 )


**Độ phức tạp Cách 3:** **Time `O(m)`**, **Space `O(m)`** (số cặp khác nhau tối đa `m`). Ưu điểm: không phụ thuộc số màu `11`.

### Cách 4 — Số quả cùng màu **lớn nhất** của từng người

**Giải thích.** Gom theo người: `by_player[x][y] += 1` (`defaultdict(Counter)`). Người `i` thắng ⇔ `max(số_lượng các màu của i) > i`. Duyệt `i` từ `0` tới `n - 1` (`default=0` cho người không nhặt gì, vì `0 > i` luôn sai kể cả `i = 0`). Cách này đọc rất giống mô tả của đề: "có màu nào nhiều hơn `i` quả".

In [14]:
def winning_player_count_max(n: int, pick: list[list[int]]) -> int:
    by_player = defaultdict(Counter)
    for x, y in pick:
        by_player[x][y] += 1
    ans = 0
    for i in range(n):
        best = max(by_player[i].values(), default=0)     # số quả cùng màu nhiều nhất của người i
        if best > i:
            ans += 1
    return ans


for n, pick, expected in examples:
    print(winning_player_count_max(n, pick), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
0 (kỳ vọng 0 )
1 (kỳ vọng 1 )


**Độ phức tạp Cách 4:** **Time `O(m + n)`**, **Space `O(m)`**.

### Cách 5 — Mảng cờ `won[]` thay `set`

**Giải thích.** Thay `set` bằng danh sách `won = [False] * n`: khi `cnt[x][y] > x` gán `won[x] = True`. Cuối cùng `sum(won)`. Về mặt ý tưởng giống Cách 2 nhưng dùng **mảng chỉ số theo người** (nhanh và gọn khi mã người liên tục `0..n-1`).

In [15]:
def winning_player_count_flags(n: int, pick: list[list[int]]) -> int:
    cnt = [[0] * 11 for _ in range(n)]
    won = [False] * n
    for x, y in pick:
        cnt[x][y] += 1
        if cnt[x][y] > x:
            won[x] = True                  # đặt lại True nhiều lần cũng không sao
    return sum(won)


for n, pick, expected in examples:
    print(winning_player_count_flags(n, pick), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
0 (kỳ vọng 0 )
1 (kỳ vọng 1 )


**Độ phức tạp Cách 5:** **Time `O(m + n · M)`**, **Space `O(n · M)`**.

## 8. TRACE từng bước

Ta trace Cách 2 (bảng + `set`) trên Ví dụ 1; Cách 4 (lớn nhất theo người) trên Ví dụ 3; và một ví dụ tự dựng `n = 3`, `pick = [[0,5],[0,7],[2,1],[2,1],[2,1],[1,2]]` đủ "khó": người `0` đạt điều kiện **hai lần** (hai màu) nhưng chỉ được tính **một** người. Đáp án tự dựng được đối chiếu với Cách 1.

In [16]:
def trace_table(n: int, pick: list[list[int]]) -> int:
    print(f"n = {n}, pick = {pick}")
    cnt = [[0] * 11 for _ in range(n)]
    s = set()
    for idx, (x, y) in enumerate(pick):
        cnt[x][y] += 1
        added = cnt[x][y] > x
        if added:
            s.add(x)
        print(f"  #{idx} [{x},{y}]: cnt[{x}][{y}] = {cnt[x][y]}  (> {x}? {added})  s = {sorted(s)}")
    print("  đáp án = len(s) =", len(s))
    return len(s)


assert trace_table(4, [[0, 0], [1, 0], [1, 0], [2, 1], [2, 1], [2, 0]]) == 2
assert trace_table(3, [[0, 5], [0, 7], [2, 1], [2, 1], [2, 1], [1, 2]]) == 2

n = 4, pick = [[0, 0], [1, 0], [1, 0], [2, 1], [2, 1], [2, 0]]
  #0 [0,0]: cnt[0][0] = 1  (> 0? True)  s = [0]
  #1 [1,0]: cnt[1][0] = 1  (> 1? False)  s = [0]
  #2 [1,0]: cnt[1][0] = 2  (> 1? True)  s = [0, 1]
  #3 [2,1]: cnt[2][1] = 1  (> 2? False)  s = [0, 1]
  #4 [2,1]: cnt[2][1] = 2  (> 2? False)  s = [0, 1]
  #5 [2,0]: cnt[2][0] = 1  (> 2? False)  s = [0, 1]
  đáp án = len(s) = 2
n = 3, pick = [[0, 5], [0, 7], [2, 1], [2, 1], [2, 1], [1, 2]]
  #0 [0,5]: cnt[0][5] = 1  (> 0? True)  s = [0]
  #1 [0,7]: cnt[0][7] = 1  (> 0? True)  s = [0]
  #2 [2,1]: cnt[2][1] = 1  (> 2? False)  s = [0]
  #3 [2,1]: cnt[2][1] = 2  (> 2? False)  s = [0]
  #4 [2,1]: cnt[2][1] = 3  (> 2? True)  s = [0, 2]
  #5 [1,2]: cnt[1][2] = 1  (> 1? False)  s = [0, 2]
  đáp án = len(s) = 2


In [17]:
def trace_max(n: int, pick: list[list[int]]) -> int:
    print(f"n = {n}, pick = {pick}")
    by_player = defaultdict(Counter)
    for x, y in pick:
        by_player[x][y] += 1
    ans = 0
    for i in range(n):
        best = max(by_player[i].values(), default=0)
        ok = best > i
        ans += ok
        print(f"  người {i}: màu->số quả = {dict(by_player[i])}, lớn nhất = {best}, cần > {i} -> {'thắng' if ok else 'không'}, ans = {ans}")
    return ans


assert trace_max(5, [[1, 1], [2, 4], [2, 4], [2, 4]]) == 1
assert winning_player_count_brute(3, [[0, 5], [0, 7], [2, 1], [2, 1], [2, 1], [1, 2]]) == 2

n = 5, pick = [[1, 1], [2, 4], [2, 4], [2, 4]]
  người 0: màu->số quả = {}, lớn nhất = 0, cần > 0 -> không, ans = 0
  người 1: màu->số quả = {1: 1}, lớn nhất = 1, cần > 1 -> không, ans = 0
  người 2: màu->số quả = {4: 3}, lớn nhất = 3, cần > 2 -> thắng, ans = 1
  người 3: màu->số quả = {}, lớn nhất = 0, cần > 3 -> không, ans = 1
  người 4: màu->số quả = {}, lớn nhất = 0, cần > 4 -> không, ans = 1


Bảng trace Cách 2 với Ví dụ 1 (`n = 4`):

| lần | `[x, y]` | `cnt[x][y]` | `> x`? | `s` sau |
|---|---|---|---|---|
| 0 | [0, 0] | 1 | có (`1 > 0`) | {0} |
| 1 | [1, 0] | 1 | không (`1 > 1` sai) | {0} |
| 2 | [1, 0] | 2 | có (`2 > 1`) | {0, 1} |
| 3 | [2, 1] | 1 | không | {0, 1} |
| 4 | [2, 1] | 2 | không (`2 > 2` sai) | {0, 1} |
| 5 | [2, 0] | 1 | không | {0, 1} |

`len(s) = 2`. Trong ví dụ tự dựng, người `0` đạt điều kiện ở **cả hai** lần nhặt (`[0,5]` và `[0,7]`) nhưng `set` giữ một bản `0` nên tổng chỉ `2` người (`0` và `2`).

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (bảng đếm + `set`): ngắn, `O(m)` sau khi tạo bảng nhỏ.

In [18]:
class Solution:
    def winningPlayerCount(self, n: int, pick: list[list[int]]) -> int:
        cnt = [[0] * 11 for _ in range(n)]      # cnt[x][y]: số quả màu y của người x (màu 0..10)
        winners = set()                          # set để mỗi người chỉ được đếm một lần
        for x, y in pick:
            cnt[x][y] += 1
            if cnt[x][y] > x:                    # nhiều hơn x quả cùng màu
                winners.add(x)
        return len(winners)


sol = Solution()
for n, pick, expected in examples:
    print(n, pick, "->", sol.winningPlayerCount(n, pick))

4 [[0, 0], [1, 0], [1, 0], [2, 1], [2, 1], [2, 0]] -> 2
5 [[1, 1], [1, 2], [1, 3], [1, 4]] -> 0
5 [[1, 1], [2, 4], [2, 4], [2, 4]] -> 1


## 10. Vì sao thuật toán đúng

- **Đúng định nghĩa.** Người `x` thắng ⇔ tồn tại màu `y` với `số_lần(x, y) > x`. Ta duy trì `cnt[x][y]` đúng bằng số lần cặp `(x, y)` đã xuất hiện trong phần đã duyệt của `pick`.
- **Đếm đúng thời điểm.** Số lần chỉ **tăng**. Nên nếu tại một thời điểm có `cnt[x][y] > x` thì điều đó **đúng mãi** về sau; ngược lại nếu cuối cùng tồn tại cặp `(x, y)` với số lần `> x` thì đúng lúc số lần đó vượt `x` ta đã kiểm tra và ghi `x` vào `set`. Vậy `set` cuối cùng chứa **đúng** các người thắng.
- **Không đếm trùng.** Dùng `set` (hoặc mảng cờ) nên một người thắng ở nhiều màu hay vượt ngưỡng nhiều lần vẫn tính **một**. Đáp án đếm **người**, không đếm cặp.
- **Màu 0..10 và chỉ số người `0..n-1` nằm trong bảng.** Theo ràng buộc `0 <= x <= n - 1` và `0 <= y <= 10`, nên `cnt[x][y]` không bao giờ ra ngoài biên (bảng `n × 11`).
- **Đối chiếu độc lập.** Cách 1 (đếm trực tiếp bằng `pick.count`) được so với các cách còn lại trên hàng nghìn test ngẫu nhiên ở mục 12.

## 11. Time / Space Complexity

Gọi `m = len(pick)`, `n` là số người, `M = 11` là số màu.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn `pick.count` | `O(n · M · m)` | `O(1)` |
| 2. Bảng `cnt` + `set` | `O(m + n · M)` | `O(n · M)` |
| 3. `Counter` trên `(x, y)` | `O(m)` | `O(m)` |
| 4. Lớn nhất theo từng người | `O(m + n)` | `O(m)` |
| 5. Mảng cờ `won[]` | `O(m + n · M)` | `O(n · M)` |

Với `n <= 10`, `m <= 100`, `M = 11` mọi cách đều rất nhanh; điểm cần nhớ là **một lượt duyệt `pick`** thay vì quét lại.

## 12. Bộ test

Gồm: ví dụ đề, biên (một lần nhặt, người `n - 1` cần `n` quả, màu `10`, người không nhặt gì), phản ví dụ cho các lỗi thường gặp, đối chiếu các cách với nhau và ngẫu nhiên có seed đúng ràng buộc.

In [19]:
import random

impls = {
    "brute": winning_player_count_brute,
    "table": winning_player_count_table,
    "counter": winning_player_count_counter,
    "max": winning_player_count_max,
    "flags": winning_player_count_flags,
    "submit": Solution().winningPlayerCount,
}

# 1) Ví dụ của đề
for n, pick, expected in examples:
    for name, fn in impls.items():
        assert fn(n, pick) == expected, (name, n, pick)
print("PASS ví dụ đề")

# 2) Biên (tự dựng, kết quả suy luận tay)
edge = [
    (2, [[0, 0]], 1),                                   # người 0 thắng chỉ với 1 quả
    (2, [[1, 0]], 0),                                   # người 1 cần 2 quả cùng màu
    (2, [[1, 0], [1, 0]], 1),                           # đủ 2 quả
    (2, [[1, 0], [1, 1]], 0),                           # 2 quả khác màu
    (10, [[9, 10]] * 10, 1),                            # người 9 cần 10 quả, màu 10 (biên)
    (10, [[9, 10]] * 9, 0),                             # thiếu 1 quả
    (3, [[0, 5], [0, 7]], 1),                           # một người thắng ở hai màu: tính một lần
    (3, [[0, 1], [1, 1], [1, 1], [2, 1], [2, 1], [2, 1]], 3),   # cả ba người thắng, cùng màu 1
    (5, [[4, 0], [4, 1], [4, 2], [4, 3], [4, 4]], 0),   # tổng 5 quả nhưng khác màu
]
for n, pick, expected in edge:
    for name, fn in impls.items():
        assert fn(n, pick) == expected, (name, n, pick, expected)
print("PASS biên")

PASS ví dụ đề
PASS biên


In [20]:
# 3) Phản ví dụ cho các cách làm sai
def sai_cong_moi_lan(n, pick):
    cnt = [[0] * 11 for _ in range(n)]
    ans = 0
    for x, y in pick:
        cnt[x][y] += 1
        if cnt[x][y] > x:
            ans += 1                    # cộng mỗi lần: một người có thể bị đếm nhiều lần
    return ans

def sai_tong_so_bong(n, pick):
    total = [0] * n
    for x, y in pick:
        total[x] += 1
    return sum(1 for i in range(n) if total[i] > i)

for n, pick in [(3, [[0, 5], [0, 7]]), (5, [[1, 1], [1, 2], [1, 3], [1, 4]])]:
    print(pick, "đúng:", Solution().winningPlayerCount(n, pick), "| cộng mỗi lần:", sai_cong_moi_lan(n, pick), "| tổng số bóng:", sai_tong_so_bong(n, pick))
assert sai_cong_moi_lan(3, [[0, 5], [0, 7]]) == 2 and Solution().winningPlayerCount(3, [[0, 5], [0, 7]]) == 1
assert sai_tong_so_bong(5, [[1, 1], [1, 2], [1, 3], [1, 4]]) == 1

[[0, 5], [0, 7]] đúng: 1 | cộng mỗi lần: 2 | tổng số bóng: 1
[[1, 1], [1, 2], [1, 3], [1, 4]] đúng: 0 | cộng mỗi lần: 0 | tổng số bóng: 1


In [21]:
# 4) Ngẫu nhiên có seed, đúng ràng buộc: 2 <= n <= 10, 1 <= len(pick) <= 100, 0 <= x < n, 0 <= y <= 10
random.seed(3238)
total_winners = 0
for _ in range(4000):
    n = random.randint(2, 10)
    m = random.randint(1, 100)
    few_colors = random.random() < 0.7                       # ít màu hơn -> dễ có người thắng
    top = random.randint(0, 3) if few_colors else 10
    pick = [[random.randint(0, n - 1), random.randint(0, top)] for _ in range(m)]
    ref = winning_player_count_brute(n, pick)
    total_winners += ref
    for name, fn in impls.items():
        assert fn(n, pick) == ref, (name, n, pick)
print("PASS 4000 test ngẫu nhiên; tổng số người thắng gặp được:", total_winners)
assert total_winners > 1000

PASS 4000 test ngẫu nhiên; tổng số người thắng gặp được: 12165


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Cộng `1` mỗi khi vượt ngưỡng (đếm trùng một người)

In [22]:
def sai_dem_trung(n, pick):
    cnt = [[0] * 11 for _ in range(n)]
    ans = 0
    for x, y in pick:
        cnt[x][y] += 1
        if cnt[x][y] > x:
            ans += 1                    # SAI: người 0 nhặt 3 quả màu 5 thì cộng 3 lần
    return ans


pick = [[0, 5], [0, 5], [0, 5]]
print("sai:", sai_dem_trung(3, pick), "| đúng:", Solution().winningPlayerCount(3, pick))

sai: 3 | đúng: 1


**Vì sao sai:** đề hỏi **số người** thắng, nhưng code cộng mỗi lần một cặp `(x, y)` vượt ngưỡng; người `0` nhặt `3` quả: lần 1, 2, 3 đều `> 0` ⇒ cộng `3` lần. **Cách sửa:** dùng `set` (hoặc mảng cờ) rồi lấy `len`/`sum`.

### Lỗi 2: Dùng `>=` thay vì `>`

In [23]:
def sai_dau_bang(n, pick):
    cnt = [[0] * 11 for _ in range(n)]
    s = set()
    for x, y in pick:
        cnt[x][y] += 1
        if cnt[x][y] >= x:              # SAI: đề yêu cầu "nhiều hơn x" (> x), tức ít nhất x + 1
            s.add(x)
    return len(s)


for n, pick, expected in examples:
    print("sai:", sai_dau_bang(n, pick), "| đúng:", expected)
assert sai_dau_bang(*examples[1][:2]) != examples[1][2]

sai: 3 | đúng: 2
sai: 1 | đúng: 0
sai: 2 | đúng: 1


**Vì sao sai:** người `i` cần **nhiều hơn `i`** quả, tức ít nhất `i + 1`. Với `>=`, người `1` chỉ cần `1` quả (thay vì `2`). Ví dụ 2: người `1` nhặt `4` quả mỗi màu `1` — `1 >= 1` sai thành thắng ⇒ trả `1` thay vì `0`. **Cách sửa:** `cnt[x][y] > x`.

### Lỗi 3: Đếm tổng số bóng của người, không theo màu

In [24]:
def sai_tong_bong(n, pick):
    total = [0] * n
    for x, y in pick:
        total[x] += 1                   # SAI: bỏ qua màu
    return sum(1 for i in range(n) if total[i] > i)


print("sai:", sai_tong_bong(5, [[1, 1], [1, 2], [1, 3], [1, 4]]), "| đúng:", Solution().winningPlayerCount(5, [[1, 1], [1, 2], [1, 3], [1, 4]]))

sai: 1 | đúng: 0


**Vì sao sai:** điều kiện là "**cùng màu**". Người `1` có `4` quả nhưng mỗi quả một màu nên không thắng, vậy mà tổng `4 > 1`. **Cách sửa:** đếm theo cặp `(x, y)`.

### Lỗi 4: Tạo bảng 2 chiều bằng `[[0] * 11] * n` (các hàng dùng chung)

In [25]:
bad = [[0] * 11] * 3                    # SAI: cả 3 phần tử là CÙNG một danh sách
bad[0][5] += 1
print(bad[0][5], bad[1][5], bad[2][5])  # 1 1 1  -> cả ba hàng cùng đổi!
print(bad[0] is bad[1])                 # True

good = [[0] * 11 for _ in range(3)]     # ĐÚNG: mỗi hàng một danh sách mới
good[0][5] += 1
print(good[0][5], good[1][5], good[2][5])     # 1 0 0


def sai_bang_chung(n, pick):
    cnt = [[0] * 11] * n
    s = set()
    for x, y in pick:
        cnt[x][y] += 1
        if cnt[x][y] > x:
            s.add(x)
    return len(s)


p = [[0, 5], [1, 5]]
print("sai:", sai_bang_chung(3, p), "| đúng:", Solution().winningPlayerCount(3, p))
assert sai_bang_chung(3, p) != Solution().winningPlayerCount(3, p)

1 1 1
True
1 0 0
sai: 2 | đúng: 1


**Vì sao sai:** `[[0] * 11] * n` lặp **cùng một đối tượng** `n` lần; sửa một hàng là sửa cả `n` hàng. Người `1` thấy luôn lượt nhặt của người `0` (màu `5`) nên cộng dồn thành `2 > 1` ⇒ thắng sai. **Cách sửa:** `[[0] * 11 for _ in range(n)]`.

### Lỗi 5: Bảng màu quá nhỏ (`10` cột thay vì `11`)

In [26]:
def sai_thieu_cot(n, pick):
    cnt = [[0] * 10 for _ in range(n)]      # SAI: màu 0..10 cần 11 cột
    s = set()
    for x, y in pick:
        cnt[x][y] += 1
        if cnt[x][y] > x:
            s.add(x)
    return len(s)


try:
    sai_thieu_cot(2, [[0, 10]])
except IndexError as e:
    print("IndexError:", e)
print("đúng:", Solution().winningPlayerCount(2, [[0, 10]]))

IndexError: list index out of range
đúng: 1


**Vì sao sai:** màu nằm trong `0..10` là **11** giá trị. Chỉ số `10` cần cột thứ `11`. **Cách sửa:** `[0] * 11` (hoặc dùng `Counter`/`dict` để khỏi phụ thuộc số màu).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Đếm số lần xuất hiện theo nhóm (người, màu)**" ⇒ bảng đếm 2 chiều (khi miền nhỏ) hoặc `Counter` trên tuple.
- "**Có ít nhất một nhóm đạt ngưỡng**" + "**đếm số đối tượng**" ⇒ dùng `set`/mảng cờ để **mỗi đối tượng chỉ một lần**.
- Ngưỡng tăng theo chỉ số ("người `i` cần `> i`") ⇒ so sánh trực tiếp `cnt[x][y] > x` ngay khi cập nhật.
- Tạo bảng 2 chiều: **luôn** `[[0] * C for _ in range(R)]`; không bao giờ `[[0] * C] * R`.
- Số màu `0..10` ⇒ `11` ô. Luôn đọc kỹ **biên** của chỉ số.
- Muốn chắc chắn: viết thêm bản vét cạn (`pick.count`) để `assert` với nhiều test ngẫu nhiên.

## 15. Biến thể của bài

1. **Liệt kê** (không đếm) các người thắng theo thứ tự tăng dần.
2. **Màu nhặt nhiều nhất** của mỗi người và số quả đó.
3. **Ngưỡng tùy biến**: người `i` cần `> f(i)` quả cùng màu (ví dụ `2 * i`).
4. **Thời điểm thắng**: chỉ số lần nhặt đầu tiên mà người đó đạt điều kiện.
5. Bài liên quan: **Majority Element (169)**, **Top K Frequent Elements (347)**, **Find Players With Zero or One Losses (2225)**, **Check if the Sentence Is Pangram (1832)**.

In [27]:
def winners_list(n, pick):
    cnt = [[0] * 11 for _ in range(n)]
    won = [False] * n
    for x, y in pick:
        cnt[x][y] += 1
        if cnt[x][y] > x:
            won[x] = True
    return [i for i in range(n) if won[i]]


def top_color_per_player(n, pick):
    by_player = defaultdict(Counter)
    for x, y in pick:
        by_player[x][y] += 1
    result = []
    for i in range(n):
        if by_player[i]:
            color, count = max(by_player[i].items(), key=lambda kv: (kv[1], -kv[0]))   # hòa: lấy màu nhỏ hơn
            result.append((color, count))
        else:
            result.append(None)
    return result


def count_with_threshold(n, pick, threshold):
    cnt = Counter(map(tuple, pick))
    return len({x for (x, y), c in cnt.items() if c > threshold(x)})


def first_win_time(n, pick):
    cnt = [[0] * 11 for _ in range(n)]
    time = [None] * n
    for t, (x, y) in enumerate(pick):
        cnt[x][y] += 1
        if cnt[x][y] > x and time[x] is None:
            time[x] = t
    return time


print(winners_list(4, [[0, 0], [1, 0], [1, 0], [2, 1], [2, 1], [2, 0]]))       # [0, 1]
print(top_color_per_player(4, [[0, 0], [1, 0], [1, 0], [2, 1], [2, 1], [2, 0]]))   # [(0, 1), (0, 2), (1, 2), None]
print(count_with_threshold(4, [[0, 0], [1, 0], [1, 0], [1, 0], [2, 1]], lambda i: 2 * i))   # 2
print(first_win_time(4, [[0, 0], [1, 0], [1, 0], [2, 1], [2, 1], [2, 0]]))     # [0, 2, None, None]
assert winners_list(4, [[0, 0], [1, 0], [1, 0], [2, 1], [2, 1], [2, 0]]) == [0, 1]
assert top_color_per_player(4, [[0, 0], [1, 0], [1, 0], [2, 1], [2, 1], [2, 0]]) == [(0, 1), (0, 2), (1, 2), None]
assert count_with_threshold(4, [[0, 0], [1, 0], [1, 0], [1, 0], [2, 1]], lambda i: 2 * i) == 2
assert first_win_time(4, [[0, 0], [1, 0], [1, 0], [2, 1], [2, 1], [2, 0]]) == [0, 2, None, None]
assert count_with_threshold(4, [[0, 0], [1, 0], [1, 0], [2, 1], [2, 1], [2, 0]], lambda i: i) == 2

[0, 1]
[(0, 1), (0, 2), (1, 2), None]
2
[0, 2, None, None]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn `pick.count` | thử mọi (người, màu), đếm bằng quét | `O(n · M · m)` | `O(1)` | **kiểm chứng**, hiểu đề |
| 2. Bảng + `set` | `cnt[x][y]`, `set` người thắng | `O(m + n · M)` | `O(n · M)` | **nên nộp** |
| 3. `Counter` `(x, y)` | đếm cặp rồi lọc `c > x` | `O(m)` | `O(m)` | không biết trước số màu |
| 4. `max` theo người | `defaultdict(Counter)`, so `max > i` | `O(m + n)` | `O(m)` | đọc giống mô tả của đề |
| 5. Mảng cờ | `won[x] = True`, `sum(won)` | `O(m + n · M)` | `O(n · M)` | mã người liên tục `0..n-1` |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `Counter([1, 1, 2])`, `[[0] * 3 for _ in range(2)]`, `Solution().winningPlayerCount(3, [[0, 5], [1, 5]])`, `Solution().winningPlayerCount(3, [[2, 0], [2, 0]])`, `max(Counter([3, 3, 4]).values())`.

**Bài 2 (tính tay).** Với `n = 3`, `pick = [[2,1],[2,1],[2,3],[2,1],[0,2],[1,2]]`: lập bảng số quả mỗi màu của từng người, rồi cho biết ai thắng và đáp án.

**Bài 3 (điền TODO).** Hoàn thành `winning_player_count_todo(n, pick)` theo Cách 2 (bảng `cnt` + `set`).

In [28]:
def winning_player_count_todo(n, pick):
    # TODO 1: tạo cnt = [[0] * 11 for _ in range(n)] và s = set()
    # TODO 2: duyệt (x, y) trong pick: cnt[x][y] += 1; nếu cnt[x][y] > x thì s.add(x)
    # TODO 3: trả về len(s)
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `winning_player_count_counter2(n, pick)` chỉ dùng `Counter(map(tuple, pick))` và một `sum`: **không** tạo `set` riêng; gợi ý: tạo tập `{x for ...}` trong một dòng rồi `len`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `winners_ids(n, pick)` trả về **danh sách tăng dần** các người thắng. (b) Viết `losers_with_balls(n, pick)` trả về các người **đã nhặt ít nhất một quả nhưng không thắng**. (c) Viết `best_player_margin(n, pick)` trả về người có **"biên độ" lớn nhất** `max_cùng_màu − i` (hòa thì chọn `i` nhỏ hơn), trả `None` nếu không ai nhặt gì.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `Counter([1, 1, 2])` đếm `1` hai lần, `2` một lần. `[[0]*3 for _ in range(2)]` là hai hàng ba số `0`. Người `0` thắng chỉ cần nhặt một quả; người `2` cần `3` quả cùng màu. Bài 3: sao chép đúng cách tạo `cnt`.
- **Mức 2 (vừa):** Bài 2: người `2`: màu `1` được `3` quả, màu `3` được `1` quả ⇒ `3 > 2` thắng; người `0`: `1 > 0` thắng; người `1`: `1 > 1` sai. Bài 4: `{x for (x, y), c in counter.items() if c > x}`.
- **Mức 3 (sâu):** Bài 5(a): `sorted(...)` của tập các người thắng. 5(b): lấy tập người có xuất hiện trong `pick` rồi **trừ** tập người thắng. 5(c): `max` cùng màu của từng người với `defaultdict(Counter)`, so sánh theo khóa `(max_cùng_màu − i, −i)`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [29]:
# Bài 1
assert Counter([1, 1, 2]) == Counter({1: 2, 2: 1})
assert [[0] * 3 for _ in range(2)] == [[0, 0, 0], [0, 0, 0]]
assert Solution().winningPlayerCount(3, [[0, 5], [1, 5]]) == 1       # chỉ người 0; người 1 cần 2 quả cùng màu
assert Solution().winningPlayerCount(3, [[2, 0], [2, 0]]) == 0       # người 2 cần 3 quả
assert max(Counter([3, 3, 4]).values()) == 2

# Bài 2
n, pick = 3, [[2, 1], [2, 1], [2, 3], [2, 1], [0, 2], [1, 2]]
table = {i: Counter(y for x, y in pick if x == i) for i in range(n)}
assert table == {0: Counter({2: 1}), 1: Counter({2: 1}), 2: Counter({1: 3, 3: 1})}
assert Solution().winningPlayerCount(n, pick) == 2                   # người 0 và người 2 thắng
assert winners_list(n, pick) == [0, 2]


# Bài 3
def winning_player_count_todo_ans(n, pick):
    cnt = [[0] * 11 for _ in range(n)]
    s = set()
    for x, y in pick:
        cnt[x][y] += 1
        if cnt[x][y] > x:
            s.add(x)
    return len(s)


for n, pick, expected in examples:
    assert winning_player_count_todo_ans(n, pick) == expected


# Bài 4
def winning_player_count_counter2(n, pick):
    return len({x for (x, y), c in Counter(map(tuple, pick)).items() if c > x})


for n, pick, expected in examples:
    assert winning_player_count_counter2(n, pick) == expected


# Bài 5
def winners_ids(n, pick):
    return sorted({x for (x, y), c in Counter(map(tuple, pick)).items() if c > x})


def losers_with_balls(n, pick):
    played = {x for x, y in pick}
    return sorted(played - set(winners_ids(n, pick)))


def best_player_margin(n, pick):
    by_player = defaultdict(Counter)
    for x, y in pick:
        by_player[x][y] += 1
    best = None
    for i in range(n):
        if by_player[i]:
            margin = max(by_player[i].values()) - i
            if best is None or margin > best[0]:
                best = (margin, i)
    return None if best is None else best[1]


p1 = [[0, 0], [1, 0], [1, 0], [2, 1], [2, 1], [2, 0]]
assert winners_ids(4, p1) == [0, 1]
assert losers_with_balls(4, p1) == [2]
assert best_player_margin(4, p1) == 0 and best_player_margin(4, [[3, 1]]) == 3 and best_player_margin(3, []) is None
assert best_player_margin(5, [[1, 1], [2, 4], [2, 4], [2, 4]]) == 2     # biên độ: người 1 -> 1 - 1 = 0, người 2 -> 3 - 2 = 1
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đếm theo cặp `(người, màu)`, ghi người thắng vào `set`)

```python
class Solution:
    def winningPlayerCount(self, n: int, pick: List[List[int]]) -> int:
        cnt = [[0] * 11 for _ in range(n)]
        winners = set()
        for x, y in pick:
            cnt[x][y] += 1
            if cnt[x][y] > x:
                winners.add(x)
        return len(winners)
```

**Checklist**

- [ ] Đếm **theo màu** (cặp `(x, y)`), không phải tổng số bóng của người?
- [ ] Điều kiện là `cnt[x][y] > x` (không phải `>=`)?
- [ ] Mỗi người chỉ được đếm **một lần** (`set`/mảng cờ), kể cả thắng ở nhiều màu?
- [ ] Bảng `cnt` tạo bằng `[[0] * 11 for _ in range(n)]` (không `* n` bên ngoài), đủ `11` cột cho màu `0..10`?
- [ ] Đã thử ba ví dụ đề (`2`, `0`, `1`) và biên: người `n - 1` cần đúng `n` quả?
- [ ] Skeleton đúng `class Solution` / `def winningPlayerCount(self, n: int, pick: List[List[int]]) -> int`?